# Arabic Medical Question Classification — Advanced Baseline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

**Task:** Classify Arabic medical questions into 8 specialties.

**Metric:** Macro F1 × 100 (0–100 scale)

**Pipeline:**
1. Clean text (remove stopwords + numbers), resolve conflicts via majority vote, deduplicate majority class
2. Standardize dialect → MSA via round-trip translation (Arabic → English → Arabic)
3. AraBERT + classification head with gradual unfreezing, cosine annealing, and inverse-frequency class weights

In [ ]:
!pip install -q kagglehub transformers sentencepiece nltk

In [ ]:
# ── DO NOT MODIFY THIS CELL ──────────────────────────────────────────
import os
import kagglehub
import pandas as pd
import numpy as np

COMP_DATASET = "sattamjaltwaim/arabic-medical-questions"  # UPDATE THIS SLUG

csv_root = kagglehub.dataset_download(COMP_DATASET)

train_csv = pd.read_csv(os.path.join(csv_root, "train.csv"))
test_csv  = pd.read_csv(os.path.join(csv_root, "test.csv"))
label_map = pd.read_csv(os.path.join(csv_root, "label_map.csv"))

idx_to_name = dict(zip(label_map["label_index"], label_map["label_name"]))
NUM_CLASSES = len(idx_to_name)

print(f"Train: {train_csv.shape}  |  Test: {test_csv.shape}  |  Classes: {NUM_CLASSES}")

## 1 — Data Cleaning

Remove stopwords and numbers, resolve conflicting labels via majority vote, then **deduplicate only the majority class** while keeping duplicates in minority classes — giving smaller classes the benefit of extra training signal from their repeated samples.

In [ ]:
import re
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

arabic_stops = set(stopwords.words("arabic"))


def clean_text(text):
    text = re.sub(r"\d+", "", text)
    words = text.split()
    words = [w for w in words if w not in arabic_stops]
    return " ".join(words)


train_csv["question"] = train_csv["question"].apply(clean_text)
test_csv["question"] = test_csv["question"].apply(clean_text)
print("Stopwords and numbers removed.")

# Majority-label resolution for conflicting questions
label_counts = train_csv.groupby(["question", "label"]).size().reset_index(name="count")
majority_labels = label_counts.sort_values("count", ascending=False).drop_duplicates("question")
majority_map = majority_labels.set_index("question")["label"]

valid_mask = train_csv["label"] == train_csv["question"].map(majority_map)
train_clean = train_csv[valid_mask].copy()
print(f"Conflicting minority labels removed: {len(train_csv) - len(train_clean)} rows")

# Keep duplicates in minority classes, deduplicate the majority class
minority_class = train_clean["label"].value_counts().idxmin()
majority_class = train_clean["label"].value_counts().idxmax()

majority_df = train_clean[train_clean["label"] == majority_class].drop_duplicates(subset=["question"])
others_df = train_clean[train_clean["label"] != majority_class]
train_clean = pd.concat([majority_df, others_df]).reset_index(drop=True)

print(f"Majority class ({idx_to_name[majority_class]}): deduplicated to {len(majority_df)} rows")
print(f"Minority classes: kept duplicates ({len(others_df)} rows)")
print(f"Final training set: {len(train_clean)} rows")

In [ ]:
import matplotlib.pyplot as plt

before_counts = train_csv[valid_mask].groupby("label").size().sort_index()
after_counts = train_clean["label"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, data, title in zip(axes, [before_counts, after_counts],
                            ["Before Dedup", "After Dedup (majority trimmed, minorities kept)"]):
    bars = ax.bar(data.index, data.values)
    ax.set_xticks(data.index)
    ax.set_xticklabels([idx_to_name[i] for i in data.index], rotation=45, ha="right")
    ax.set_ylabel("Count")
    ax.set_title(title)
    for bar, v in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 10, str(v),
                ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

print("\nClass count changes:")
for cls in sorted(before_counts.index):
    b, a = before_counts[cls], after_counts[cls]
    diff = a - b
    print(f"  {idx_to_name[cls]}: {b} → {a} ({diff:+d})")

## 2 — Standardize Dialect → MSA (Round-Trip Translation)

Translate Arabic → English → Arabic. The back-translation produces Modern Standard Arabic because translation models are trained on formal text.

In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


def translate_batch(texts, model, tokenizer, batch_size=32, max_len=512):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=max_len,
        ).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, num_beams=4, max_length=max_len)
        results.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
        if (i // batch_size) % 20 == 0:
            print(f"  {min(i + batch_size, len(texts))}/{len(texts)}")
    return results

In [ ]:
train_questions = train_clean["question"].tolist()
test_questions = test_csv["question"].tolist()

# Arabic → English
print("Loading ar→en model...")
ar_en_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ar-en")
ar_en_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-ar-en").to(device)

print("Translating train → English...")
train_en = translate_batch(train_questions, ar_en_model, ar_en_tok)
print("\nTranslating test → English...")
test_en = translate_batch(test_questions, ar_en_model, ar_en_tok)

del ar_en_model, ar_en_tok
torch.cuda.empty_cache()

# English → Arabic (MSA)
print("\nLoading en→ar model...")
en_ar_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ar")
en_ar_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-ar").to(device)

print("Translating train → MSA...")
train_msa = translate_batch(train_en, en_ar_model, en_ar_tok)
print("\nTranslating test → MSA...")
test_msa = translate_batch(test_en, en_ar_model, en_ar_tok)

del en_ar_model, en_ar_tok
torch.cuda.empty_cache()

In [ ]:
for i in range(3):
    print(f"\n--- Example {i} ---")
    print(f"Original: {train_questions[i][:120]}")
    print(f"English:  {train_en[i][:120]}")
    print(f"MSA:      {train_msa[i][:120]}")

## 3 — AraBERT + Classification Head

Fine-tune [AraBERT v0.2](https://huggingface.co/aubmindlab/bert-base-arabertv02) with a classification head.

**Training strategy:**
- Epochs 0–3: only the classification head trains (AraBERT frozen)
- Epochs 4–7: unfreeze last 4 encoder layers
- Epochs 8+: unfreeze everything
- Up to 20 epochs with early stopping (patience=4 on val F1)
- Cosine annealing with warmup
- Inverse-frequency class weights in CrossEntropyLoss

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import torch.nn as nn

MODEL_NAME = "aubmindlab/bert-base-arabertv02"
MAX_LEN = 256
BATCH_SIZE = 16
MAX_EPOCHS = 20
PATIENCE = 4
HEAD_LR = 2e-4
BERT_LR = 2e-5

bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class MedicalDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = bert_tokenizer(
            texts, truncation=True, padding=True,
            max_length=MAX_LEN, return_tensors="pt",
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class ArabertClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hidden = self.bert.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 4, NUM_CLASSES),
        )

    def forward(self, input_ids, attention_mask, **kwargs):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(out.last_hidden_state[:, 0])

In [ ]:
def set_unfreezing(model, phase):
    """Gradually unfreeze AraBERT layers."""
    if phase == "head":
        for p in model.bert.parameters():
            p.requires_grad = False
    elif phase == "last_4":
        for p in model.bert.parameters():
            p.requires_grad = False
        for layer in model.bert.encoder.layer[-4:]:
            for p in layer.parameters():
                p.requires_grad = True
    elif phase == "all":
        for p in model.bert.parameters():
            p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  [{phase}] Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})")


UNFREEZE_SCHEDULE = {0: "head", 4: "last_4", 8: "all"}

## 4 — Train / Validate / Predict

In [ ]:
y_all = train_clean["label"].values

# Stratified train/val split
tr_idx, val_idx = train_test_split(
    np.arange(len(train_msa)), test_size=0.15,
    stratify=y_all, random_state=42,
)
tr_texts = [train_msa[i] for i in tr_idx]
val_texts = [train_msa[i] for i in val_idx]
tr_labels = y_all[tr_idx]
val_labels = y_all[val_idx]

print(f"Train: {len(tr_texts)}  |  Val: {len(val_texts)}")

train_ds = MedicalDataset(tr_texts, tr_labels)
val_ds = MedicalDataset(val_texts, val_labels)
test_ds = MedicalDataset(test_msa)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2)

# Inverse-frequency class weights
from collections import Counter
class_counts = Counter(tr_labels.tolist())
total = sum(class_counts.values())
class_weights = torch.tensor(
    [total / (NUM_CLASSES * class_counts[c]) for c in range(NUM_CLASSES)],
    dtype=torch.float,
).to(device)
print(f"\nClass weights:")
for c in range(NUM_CLASSES):
    print(f"  {idx_to_name[c]}: {class_weights[c]:.2f}  (n={class_counts[c]})")

# Model + optimizer + scheduler
model = ArabertClassifier().to(device)

optimizer = torch.optim.AdamW([
    {"params": model.head.parameters(), "lr": HEAD_LR},
    {"params": model.bert.parameters(), "lr": BERT_LR},
], weight_decay=0.01)

total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = len(train_loader)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_f1 = 0
best_state = None
patience_counter = 0

for epoch in range(MAX_EPOCHS):
    if epoch in UNFREEZE_SCHEDULE:
        set_unfreezing(model, UNFREEZE_SCHEDULE[epoch])

    # Train
    model.train()
    running_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    # Validate
    model.eval()
    val_preds = []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            val_preds.extend(logits.argmax(dim=1).cpu().numpy())
    val_preds = np.array(val_preds)

    f1 = f1_score(val_labels, val_preds, average="macro")
    prec = precision_score(val_labels, val_preds, average="macro")
    rec = recall_score(val_labels, val_preds, average="macro")
    lr = scheduler.get_last_lr()[0]

    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = " ★"
    else:
        patience_counter += 1
        marker = f" (no improve {patience_counter}/{PATIENCE})"

    print(f"  Epoch {epoch:2d}: loss={avg_loss:.4f}  F1={f1:.4f}  P={prec:.4f}  R={rec:.4f}  lr={lr:.2e}{marker}")

    if patience_counter >= PATIENCE:
        print(f"\n  Early stopping at epoch {epoch} — no improvement for {PATIENCE} epochs")
        break

print(f"\nBest val F1: {best_f1:.4f} (trained {epoch + 1}/{MAX_EPOCHS} epochs)")

# Restore best model
model.load_state_dict(best_state)
model.to(device)
model.eval()

print(f"\nValidation Classification Report (best checkpoint):")
val_preds_best = []
with torch.no_grad():
    for batch in val_loader:
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        val_preds_best.extend(logits.argmax(dim=1).cpu().numpy())

target_names = [f"{i}: {idx_to_name[i]}" for i in range(NUM_CLASSES)]
print(classification_report(val_labels, val_preds_best, target_names=target_names, digits=4))

## 5 — Test Prediction & Submission

In [ ]:
all_preds = []
with torch.no_grad():
    for batch in test_loader:
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())

all_preds = np.array(all_preds)
print(f"Test predictions: {len(all_preds)} questions")
for cls in range(NUM_CLASSES):
    n = (all_preds == cls).sum()
    print(f"  {cls} ({idx_to_name[cls]}): {n}")

In [ ]:
# ── DO NOT MODIFY THIS CELL ──────────────────────────────────────────
def generate_submission(predictions, filename="submission.csv"):
    """Create a Kaggle submission file from predictions."""
    y = np.asarray(predictions, dtype=int)
    assert len(y) == len(test_csv), (
        f"Expected {len(test_csv)} predictions, got {len(y)}"
    )
    submission = pd.DataFrame({
        "id": test_csv["id"].values,
        "prediction": y.astype(float),
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename}  ({len(submission)} rows)")
    print(submission.head())

generate_submission(all_preds)